# Review 2: full-data continual Task 1 foundation

Run on Kaggle with Internet and a T4 GPU. Attach the **existing prepared folder** as a private Kaggle input, or reuse it in `/kaggle/working`; set `PREPARED_DIR` below. This is a fresh five-output model for global IDs 0–4, not the eight-class static checkpoint. We train only Task 1 here. Task 2 training, replay comparisons, and poisoning are later runs. The combined flow collection has unverified per-row source provenance.


## 1. Check out the published API and verify the runtime

The exact source commit contains disk-backed training, saved-result plots, novelty metrics, and their synthetic checks. Keep Kaggle's CUDA-enabled PyTorch; do not install the local requirements file over it.


In [ ]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, shutil, platform, re, random
from dataclasses import asdict
from datetime import datetime, timezone
from importlib.metadata import version
from time import perf_counter
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = 'e6ebfcf7a9beec4d2b8fd5245e7c62314b9fadf9'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-full-task1')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.data import BROAD_CLASS_ORDER, FittedPreprocessor
from src.data.large_data import large_data_paths, verify_full_partitions
from src.training.disk_backed import DiskBackedFlowDataset
from src.continual_learning.disk_replay import select_disk_replay_rows
from src.training.full_run import train_full_disk_backed
from src.evaluation.streaming import ConfusionAccumulator
from src.evaluation.novelty import novelty_metrics
from src.evaluation.plots import save_result_plots
from src.novelty import detect_unknown
from src.models import ModelConfig, TabularTransformerClassifier
if not torch.cuda.is_available():
    raise RuntimeError('CUDA required: enable Kaggle T4 GPU and restart')
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
environment = {'python': platform.python_version(), 'numpy': version('numpy'), 'torch': torch.__version__, 'cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0), 'cuda_device_count': torch.cuda.device_count()}
print('Verified source:', checked_out_sha, 'environment:', environment)
for pattern in ('test_full_disk_training.py', 'test_large_data.py', 'test_novelty_evaluation.py', 'test_result_plots.py', 'test_disk_replay.py'):
    result = subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-p', pattern, '-v'], cwd=REPO_DIR, check=True, capture_output=True, text=True)
    print(result.stdout, end=''); print(result.stderr, end='')
    summary = result.stdout + result.stderr
    count = re.search(r'Ran (\d+) tests? in', summary)
    if count is None or int(count.group(1)) == 0 or re.search(r'OK \(skipped=\d+\)', summary):
        raise RuntimeError(f'Focused tests absent or skipped: {pattern}')


## 2. Select and audit the existing prepared collection

`PREPARED_DIR` must point to the folder containing the four row-aligned arrays, split files, manifests, and `preprocessing_continual.json`. This stage reads only: it checks the original input checksum, source, split coverage, 54-feature order, target exclusions, and the Task-1-only fit scope. It does **not** prepare data again or use `preprocessing_static.json` for training.


In [ ]:
PREPARED_DIR = None  # set to '/kaggle/input/.../review2_large_prepare_...' or '/kaggle/working/review2_large_prepare_...'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
EXPECTED_PREP_SOURCE_SHA = '5c0d519596bc6c4b99948126443635c2f7eb15f1'
EXPECTED_ROWS = 9_167_581
TASK1_IDS = (0, 1, 2, 3, 4)
TASK2_IDS = (5, 6, 7)
EXPECTED_TASK1_TRAIN = 6_347_232
if PREPARED_DIR is None:
    candidates = [p.parent for base in (Path('/kaggle/working'), Path('/kaggle/input')) for p in base.rglob('manifest.json') if 'review2_large_prepare_' in str(p.parent)]
    print('Prepared candidates:', [str(p) for p in candidates])
    if len(candidates) != 1:
        raise RuntimeError(f'Found {len(candidates)} prepared folders; set PREPARED_DIR explicitly')
    prepared_dir = candidates[0]
else:
    prepared_dir = Path(PREPARED_DIR)
if not prepared_dir.is_dir() or not (prepared_dir.is_relative_to(Path('/kaggle/working')) or prepared_dir.is_relative_to(Path('/kaggle/input'))):
    raise ValueError('PREPARED_DIR must be a Kaggle working/input folder')
prep_manifest = json.loads((prepared_dir / 'manifest.json').read_text())
split_manifest = json.loads((prepared_dir / 'split_manifest.json').read_text())
features_manifest = json.loads((prepared_dir / 'features_manifest.json').read_text())
continual_record = json.loads((prepared_dir / 'preprocessing_continual.json').read_text())
static_record = json.loads((prepared_dir / 'preprocessing_static.json').read_text())
assert prep_manifest['completed_preparation_and_benchmark_only'] is True
assert prep_manifest['source_sha'] == EXPECTED_PREP_SOURCE_SHA and prep_manifest['input_sha256'] == EXPECTED_INPUT_SHA256
assert prep_manifest['input_rows'] == split_manifest['row_count'] == features_manifest['row_count'] == EXPECTED_ROWS
assert split_manifest['seed'] == 42 and split_manifest['class_order'] == list(BROAD_CLASS_ORDER)
assert split_manifest['fractions'] == [0.70, 0.15, 0.15] and split_manifest['split_codes'] == {'train': 0, 'validation': 1, 'test': 2}
assert features_manifest['dtype'] == 'float32' and features_manifest['targets_excluded'] == ['Label', 'ClassLabel']
assert len(features_manifest['feature_names']) == 54 and features_manifest['feature_names'] == prep_manifest['stages']['feature_store']['feature_names']
assert continual_record['scope'] == 'continual' and static_record['scope'] == 'static'
assert continual_record['feature_names'] == static_record['feature_names'] == features_manifest['feature_names']
assert continual_record['fit_rows'] == EXPECTED_TASK1_TRAIN and continual_record['reservoir_seed'] == 42 and continual_record['reservoir_size_used'] == 200_000
assert continual_record['fit_rows'] == sum(split_manifest['counts'][BROAD_CLASS_ORDER[class_id]]['train'] for class_id in TASK1_IDS)
paths = large_data_paths(prepared_dir)
actual_split = verify_full_partitions(paths, EXPECTED_ROWS)
assert actual_split['counts'] == split_manifest['counts']
for filename, dtype, shape in (('raw_features.npy', 'float32', (EXPECTED_ROWS, 54)), ('labels.npy', 'uint8', (EXPECTED_ROWS,)), ('splits.npy', 'uint8', (EXPECTED_ROWS,)), ('row_ids.npy', 'int64', (EXPECTED_ROWS,))):
    array = np.load(prepared_dir / filename, mmap_mode='r', allow_pickle=False)
    assert array.dtype == np.dtype(dtype) and array.shape == shape
    assert (prepared_dir / filename).stat().st_size == prep_manifest['artifact_bytes'][filename]
for partition in ('train', 'validation', 'test'):
    size = sum(counts[partition] for counts in split_manifest['counts'].values())
    indices = np.load(prepared_dir / f'{partition}_indices.npy', mmap_mode='r', allow_pickle=False)
    assert indices.dtype == np.int64 and indices.shape == (size,)
continual_state = FittedPreprocessor(tuple(continual_record['feature_names']), np.asarray(continual_record['imputation_values'], dtype=np.float64), np.asarray(continual_record['means'], dtype=np.float64), np.asarray(continual_record['scales'], dtype=np.float64))
assert all(np.isfinite(values).all() for values in (continual_state.imputation_values, continual_state.means, continual_state.scales)) and np.all(continual_state.scales > 0)
print('Validated read-only prepared folder:', prepared_dir, 'Task 1 train:', EXPECTED_TASK1_TRAIN)


## 3. Bind data/preprocessing identity and choose a durable output folder

Hash the prepared bytes and frozen continual fit state before training. `RESUME_FROM` can be a saved run in `/kaggle/working` or an attached prior Kaggle output; attached inputs are copied to a new writable folder. Resume rejects changed data, configuration, and runtime. Save the entire run folder as a private Kaggle output/version after each session, including both checkpoints. An interrupted *partial* epoch repeats from the last completed epoch.


In [ ]:
RESUME_FROM = None  # optional previous '/kaggle/working/full_task1_...' or attached '/kaggle/input/.../full_task1_...'
MAX_EPOCHS_THIS_CALL = None  # optional positive integer for a planned epoch-boundary pause
def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()
identity_start = perf_counter()
data_identity = {'prep_source_sha': prep_manifest['source_sha'], 'input_sha256': prep_manifest['input_sha256'], 'row_count': EXPECTED_ROWS, 'split_seed': 42, 'raw_features_sha256': file_sha256(paths.raw_features), 'labels_sha256': file_sha256(paths.labels), 'splits_sha256': file_sha256(paths.splits), 'row_ids_sha256': file_sha256(paths.row_ids), 'split_manifest_sha256': file_sha256(prepared_dir / 'split_manifest.json'), 'continual_preprocessing_sha256': file_sha256(prepared_dir / 'preprocessing_continual.json'), 'preprocessing_scope': 'Task 1 training global IDs 0-4 only', 'task1_class_ids': list(TASK1_IDS), 'task1_train_rows': EXPECTED_TASK1_TRAIN}
print('Prepared bytes hashed in seconds:', perf_counter() - identity_start)
if RESUME_FROM is None:
    run_dir = Path('/kaggle/working') / ('full_task1_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    run_dir.mkdir(exist_ok=False)
    resume = False
else:
    previous_dir = Path(RESUME_FROM)
    if not previous_dir.is_dir() or not (previous_dir.is_relative_to(Path('/kaggle/working')) or previous_dir.is_relative_to(Path('/kaggle/input'))):
        raise ValueError('RESUME_FROM must be an existing Kaggle run folder')
    if previous_dir.is_relative_to(Path('/kaggle/working')):
        run_dir = previous_dir
    else:
        run_dir = Path('/kaggle/working') / ('full_task1_resume_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
        run_dir.mkdir(exist_ok=False)
        for name in ('latest.pt', 'best.pt', 'preprocessing_continual.json', 'history.json', 'training_manifest.json', 'run_config.json', 'run_environment.json', 'reload_verification.json', 'task1_test_metrics.json', 'novelty.json', 'replay_buffer.npz', 'replay_manifest.json', 'final_manifest.json', 'training_curves.png', 'task1_confusion_counts.png', 'task1_confusion_row_normalized.png', 'task1_per_class_recall_f1.png'):
            if (previous_dir / name).exists():
                shutil.copy2(previous_dir / name, run_dir / name)
    resume = True
assert run_dir != prepared_dir and not run_dir.is_relative_to(prepared_dir)
if shutil.disk_usage('/kaggle/working').free < 1_000_000_000:
    raise RuntimeError('Less than 1 GB free for checkpoints and output')
state_copy = run_dir / 'preprocessing_continual.json'
if state_copy.exists():
    assert file_sha256(state_copy) == data_identity['continual_preprocessing_sha256']
else:
    shutil.copy2(prepared_dir / 'preprocessing_continual.json', state_copy)
print('Run folder:', run_dir, 'resume:', resume)


## 4. Train a fresh five-class model on all Task 1 training rows

The disk view selects only global IDs 0–4 within the original training split; the validation view also contains only these old classes. Features are transformed in bounded batches by the frozen `preprocessing_continual.json` state. The model is initialized from seed 42, never loaded from the eight-class static checkpoint. Full Task 1 validation loss alone chooses the best epoch.


In [ ]:
config = ModelConfig(num_features=54, num_classes=5, hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=256, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42, training_sampler='shuffled')
train_view = DiskBackedFlowDataset(prepared_dir, continual_state, partition='train', class_ids=TASK1_IDS)
validation_view = DiskBackedFlowDataset(prepared_dir, continual_state, partition='validation', class_ids=TASK1_IDS)
task1_validation_rows = sum(split_manifest['counts'][BROAD_CLASS_ORDER[class_id]]['validation'] for class_id in TASK1_IDS)
assert len(train_view) == EXPECTED_TASK1_TRAIN and len(validation_view) == task1_validation_rows
assert not np.intersect1d(train_view.indices, validation_view.indices).size
assert np.isin(train_view.labels[train_view.indices], TASK1_IDS).all() and np.isin(validation_view.labels[validation_view.indices], TASK1_IDS).all()
data_identity['task1_validation_rows'] = task1_validation_rows
config_record = {'source_sha': checked_out_sha, 'model': asdict(config), 'class_ids': list(TASK1_IDS), 'preprocessing_scope': 'continual Task 1 training only', 'data_identity': data_identity, 'checkpoint_selection': 'minimum full Task 1 validation loss only'}
config_path = run_dir / 'run_config.json'
if config_path.exists():
    assert json.loads(config_path.read_text()) == config_record
else:
    config_path.write_text(json.dumps(config_record, indent=2))
env_path = run_dir / 'run_environment.json'
if env_path.exists():
    assert json.loads(env_path.read_text()) == environment
else:
    env_path.write_text(json.dumps(environment, indent=2))
random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)
model = TabularTransformerClassifier(config, class_ids=TASK1_IDS)
assert model.class_ids == TASK1_IDS
training_result = train_full_disk_backed(model, train_view, validation_view, output_dir=run_dir, data_identity=data_identity, resume=resume, max_epochs_this_call=MAX_EPOCHS_THIS_CALL, progress_every_batches=1000, report=lambda message: print(message, flush=True))
print('Epochs:', training_result['completed_epoch'], 'best:', training_result['best_epoch'], 'optimizer steps:', training_result['history']['optimizer_steps'])
if training_result['paused']:
    raise RuntimeError('Paused at a completed epoch. Save the entire run folder and rerun with RESUME_FROM before test evaluation.')


## 5. Reload the validation-selected Task 1 checkpoint

Compare the loadable best checkpoint against the authoritative best weights inside `latest.pt` and a second reload on a bounded validation batch. This checks persistence; it does not open the test split. Plot measured training and Task-1-only validation history.


In [ ]:
best_path = run_dir / 'best.pt'
latest = torch.load(run_dir / 'latest.pt', map_location='cpu', weights_only=False)
assert latest['data_identity'] == data_identity and latest['best_epoch'] == training_result['best_epoch']
best_model = TabularTransformerClassifier.load(best_path, device='cuda')
second_load = TabularTransformerClassifier.load(best_path, device='cuda')
assert best_model.class_ids == second_load.class_ids == TASK1_IDS
assert all(torch.equal(tensor.detach().cpu(), latest['best_state_dict'][name]) for name, tensor in best_model.network.state_dict().items())
sample_x, _, _ = validation_view.batch(validation_view.indices[:256])
assert np.array_equal(best_model.predict(sample_x), second_load.predict(sample_x))
reload_record = {'best_epoch': training_result['best_epoch'], 'latest_best_state_equal': True, 'bounded_validation_predictions_identical': True, 'best_checkpoint_sha256': file_sha256(best_path)}
reload_path = run_dir / 'reload_verification.json'
if reload_path.exists():
    assert json.loads(reload_path.read_text()) == reload_record
else:
    reload_path.write_text(json.dumps(reload_record, indent=2))
rows = training_result['history']['epochs']
if not (run_dir / 'training_curves.png').exists():
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot([row['epoch'] for row in rows], [row['train_loss'] for row in rows], label='Train')
    axes[0].plot([row['epoch'] for row in rows], [row['validation_loss'] for row in rows], label='Task 1 validation')
    axes[0].axvline(training_result['best_epoch'], color='gray', linestyle='--', label='Best')
    axes[0].set(xlabel='Epoch', ylabel='Cross-entropy', title='Task 1 loss'); axes[0].legend()
    axes[1].plot([row['epoch'] for row in rows], [row['validation_accuracy'] for row in rows], marker='o')
    axes[1].set(xlabel='Epoch', ylabel='Accuracy', title='Task 1 validation accuracy')
    fig.tight_layout(); fig.savefig(run_dir / 'training_curves.png', dpi=160); plt.show()
print('Best reload verified:', reload_record)


## 6. Freeze a clean 500-row replay buffer for a later Task 2 run

With seed 42, sample 100 unique **training** rows from each old class without replacement. Store original dataset row IDs, global labels, and frozen transformed float32 features. The buffer is selected but **not used to train Task 2 here**. Split-code checks explicitly exclude validation/test.


In [ ]:
replay_path = run_dir / 'replay_buffer.npz'
replay_manifest_path = run_dir / 'replay_manifest.json'
selected_positions = select_disk_replay_rows(train_view, class_ids=TASK1_IDS, per_class=100, seed=42)
assert len(selected_positions) == len(np.unique(selected_positions)) == 500
assert np.all(train_view.splits[selected_positions] == 0) and np.isin(train_view.labels[selected_positions], TASK1_IDS).all()
assert not np.intersect1d(selected_positions, validation_view.indices).size
replay_x, replay_y, replay_original_ids = train_view.batch(selected_positions)
assert replay_x.shape == (500, 54) and replay_x.dtype == np.float32 and np.isfinite(replay_x).all()
assert np.array_equal(replay_original_ids, selected_positions) and all(np.count_nonzero(replay_y == class_id) == 100 for class_id in TASK1_IDS)
replay_record = {'seed': 42, 'class_ids': list(TASK1_IDS), 'rows_per_class': 100, 'unique_rows': 500, 'source_partition': 'original training only', 'preprocessing_sha256': data_identity['continual_preprocessing_sha256'], 'counts': {str(class_id): int(np.count_nonzero(replay_y == class_id)) for class_id in TASK1_IDS}, 'original_row_ids': replay_original_ids.tolist()}
if replay_path.exists() or replay_manifest_path.exists():
    assert replay_path.exists() and replay_manifest_path.exists() and json.loads(replay_manifest_path.read_text()) == replay_record
    with np.load(replay_path, allow_pickle=False) as saved:
        assert np.array_equal(saved['features'], replay_x) and np.array_equal(saved['labels'], replay_y) and np.array_equal(saved['original_row_ids'], replay_original_ids)
else:
    np.savez_compressed(replay_path, features=replay_x, labels=replay_y, original_row_ids=replay_original_ids)
    replay_manifest_path.write_text(json.dumps(replay_record, indent=2))
print('Frozen replay buffer: 500 unique Task 1 training rows, 100 per class')


## 7. Calibrate novelty on old-class validation, then open held-out test flows

Only Task 1 validation maximum probabilities determine the fixed fifth-percentile confidence threshold. `unknown` means maximum probability **strictly below** that threshold. After calibration, bounded batches evaluate Task 1 known test and unseen Task 2 test flows with the unchanged five-output head. The same known-test pass also accumulates Task 1 classification confusion, avoiding repeat test inference. We do not fit preprocessing or tune the threshold from Task 2. Confidence thresholding is a baseline and may have poor unknown recall.


In [ ]:
task1_metrics_path = run_dir / 'task1_test_metrics.json'
novelty_path = run_dir / 'novelty.json'
if task1_metrics_path.exists() or novelty_path.exists():
    assert task1_metrics_path.exists() and novelty_path.exists(), 'Incomplete saved test records; inspect before rerun'
    task1_record = json.loads(task1_metrics_path.read_text())
    novelty_record = json.loads(novelty_path.read_text())
    assert task1_record['data_identity'] == novelty_record['data_identity'] == data_identity
    assert task1_record['best_checkpoint_sha256'] == novelty_record['best_checkpoint_sha256'] == reload_record['best_checkpoint_sha256']
    print('Reusing matching saved Task 1/novelty test records; no new test inference')
else:
    validation_confidences = []
    for start in range(0, len(validation_view), config.batch_size):
        x, _, _ = validation_view.batch(validation_view.indices[start:start + config.batch_size])
        probabilities = best_model.predict_proba(x)
        assert probabilities.shape == (len(x), 5) and np.isfinite(probabilities).all()
        validation_confidences.append(probabilities.max(axis=1))
    validation_confidences = np.concatenate(validation_confidences)
    assert len(validation_confidences) == task1_validation_rows
    threshold = float(np.quantile(validation_confidences, 0.05, method='linear'))
    assert np.isfinite(threshold) and 0 <= threshold <= 1
    calibration = {'threshold': threshold, 'quantile': 0.05, 'method': 'linear', 'fit_scope': 'Task 1 validation only', 'validation_rows': task1_validation_rows}
    known_view = DiskBackedFlowDataset(prepared_dir, continual_state, partition='test', class_ids=TASK1_IDS)
    unknown_view = DiskBackedFlowDataset(prepared_dir, continual_state, partition='test', class_ids=TASK2_IDS)
    confusion = ConfusionAccumulator(TASK1_IDS)
    known_flags, unknown_flags, unknown_ids = [], [], []
    for start in range(0, len(known_view), config.batch_size):
        x, truth, _ = known_view.batch(known_view.indices[start:start + config.batch_size])
        probabilities = best_model.predict_proba(x)
        predicted = np.asarray(best_model.class_ids)[probabilities.argmax(axis=1)]
        confusion.update(truth, predicted)
        known_flags.append(detect_unknown(probabilities, threshold))
    for start in range(0, len(unknown_view), config.batch_size):
        x, truth, _ = unknown_view.batch(unknown_view.indices[start:start + config.batch_size])
        probabilities = best_model.predict_proba(x)
        unknown_flags.append(detect_unknown(probabilities, threshold))
        unknown_ids.append(truth)
    known_flags = np.concatenate(known_flags); unknown_flags = np.concatenate(unknown_flags); unknown_ids = np.concatenate(unknown_ids)
    assert len(known_flags) == len(known_view) and len(unknown_flags) == len(unknown_view)
    metrics = confusion.metrics(class_names=dict(enumerate(BROAD_CLASS_ORDER)), benign_class_id=0)
    assert metrics['total_rows'] == len(known_view)
    truth_flags = np.concatenate((np.zeros(len(known_flags), dtype=bool), np.ones(len(unknown_flags), dtype=bool)))
    predicted_flags = np.concatenate((known_flags, unknown_flags))
    true_ids = np.concatenate((known_view.labels[known_view.indices].astype(np.int64), unknown_ids))
    novelty = novelty_metrics(truth_flags, predicted_flags, true_class_ids=true_ids, unknown_class_ids=TASK2_IDS)
    task1_record = {'data_identity': data_identity, 'best_checkpoint_sha256': reload_record['best_checkpoint_sha256'], 'best_validation_epoch': training_result['best_epoch'], 'metrics': metrics}
    novelty_record = {'data_identity': data_identity, 'best_checkpoint_sha256': reload_record['best_checkpoint_sha256'], 'class_ids_in_task1_head': list(TASK1_IDS), 'held_out_class_ids': list(TASK2_IDS), 'calibration': calibration, 'metrics': novelty, 'protocol': 'known Task 1 test versus held-out Task 2 test before head expansion'}
    task1_metrics_path.write_text(json.dumps(task1_record, indent=2))
    novelty_path.write_text(json.dumps(novelty_record, indent=2))
plot_names = ('task1_confusion_counts.png', 'task1_confusion_row_normalized.png', 'task1_per_class_recall_f1.png')
existing_plots = [(run_dir / name).exists() for name in plot_names]
if any(existing_plots) and not all(existing_plots):
    raise RuntimeError('Incomplete plot outputs; inspect the run folder before rerun')
if not any(existing_plots):
    print('Saved count-derived plots:', save_result_plots(task1_metrics_path, run_dir, prefix='task1'))
else:
    print('Reusing three existing saved-result plots')
print('Task 1 old-test accuracy:', task1_record['metrics']['accuracy'], 'macro-F1:', task1_record['metrics']['macro_f1'])
print('Held-out novelty:', novelty_record['metrics'])


## 8. Final audit and preservation

The folder contains the five-class Task 1 foundation for a later Task 2 comparison: best/latest checkpoints, history, hashes, replay features/IDs, calibration, old-class and held-out novelty metrics, and plots. Save the **whole `/kaggle/working/full_task1_...` folder** as a Kaggle version or private dataset; attach it alongside the same prepared folder in the next session. Do not commit arrays or checkpoints to Git. Static clean test results are only a separate reference; they did not set Task 1 hyperparameters or novelty threshold.


In [ ]:
final_manifest = {'status': 'completed full-data Task 1 foundation only', 'source_sha': checked_out_sha, 'prepared_dir': str(prepared_dir), 'data_identity': data_identity, 'class_ids': list(TASK1_IDS), 'training_rows': len(train_view), 'validation_rows': len(validation_view), 'completed_epoch': training_result['completed_epoch'], 'best_epoch': training_result['best_epoch'], 'best_validation_loss': training_result['best_validation_loss'], 'optimizer_steps': training_result['history']['optimizer_steps'], 'replay_unique_training_rows': replay_record['unique_rows'], 'best_checkpoint_sha256': reload_record['best_checkpoint_sha256'], 'old_test_rows': task1_record['metrics']['total_rows'], 'novelty_known_rows': novelty_record['metrics']['known_true'], 'novelty_unknown_rows': novelty_record['metrics']['unknown_true'], 'task2_trained': False, 'artifact_names': sorted(path.name for path in run_dir.iterdir() if path.is_file())}
(run_dir / 'final_manifest.json').write_text(json.dumps(final_manifest, indent=2))
print('Save entire Kaggle run folder:', run_dir)
print('Return executed notebook and final_manifest.json, training_manifest.json, history.json, run_config.json, run_environment.json, preprocessing_continual.json, reload_verification.json, replay_manifest.json, task1_test_metrics.json, novelty.json, training_curves.png, all three task1 plot PNGs. Preserve replay_buffer.npz, best.pt, latest.pt privately for Task 2 and resume.')
